# DATA CLEANING & PROCESSING

*The goal of this phase was to transform the raw bank marketing dataset into a clean, consistent format suitable for Exploratory Data Analysis (EDA). The process focused on handling missing values, standardizing formats, managing outliers, and engineering initial features.*

In [4]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.read_csv("C:/Users/Dell/OneDrive/Desktop/EDA PROJECT/Raw/Raw Dataset (uncleaned).csv",sep=";")

### 1.) Standardizing Categorical Columns

In [6]:
# Capitalize all string columns and trim white spaces.

for cols in df.select_dtypes(include=["object"]).columns:
    df[cols] = df[cols].str.capitalize().str.strip()
df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,Management,Married,Tertiary,No,2143,Yes,No,Unknown,5,May,261,1,-1,0,Unknown,No
1,44,Technician,Single,Secondary,No,29,Yes,No,Unknown,5,May,151,1,-1,0,Unknown,No
2,33,Entrepreneur,Married,Secondary,No,2,Yes,Yes,Unknown,5,May,76,1,-1,0,Unknown,No
3,47,Blue-collar,Married,Unknown,No,1506,Yes,No,Unknown,5,May,92,1,-1,0,Unknown,No
4,33,Unknown,Single,Unknown,No,1,No,No,Unknown,5,May,198,1,-1,0,Unknown,No


### 2.) Handiling Missing Values

In [7]:
#Replacing All Unknown To NaN for easier Processing

df.replace("Unknown",np.nan,inplace=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   age        45211 non-null  int64 
 1   job        44923 non-null  object
 2   marital    45211 non-null  object
 3   education  43354 non-null  object
 4   default    45211 non-null  object
 5   balance    45211 non-null  int64 
 6   housing    45211 non-null  object
 7   loan       45211 non-null  object
 8   contact    32191 non-null  object
 9   day        45211 non-null  int64 
 10  month      45211 non-null  object
 11  duration   45211 non-null  int64 
 12  campaign   45211 non-null  int64 
 13  pdays      45211 non-null  int64 
 14  previous   45211 non-null  int64 
 15  poutcome   8252 non-null   object
 16  y          45211 non-null  object
dtypes: int64(7), object(10)
memory usage: 5.9+ MB


In [8]:
#Droping Columns with excessive missing values

df.drop(columns=['poutcome'],inplace=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 16 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   age        45211 non-null  int64 
 1   job        44923 non-null  object
 2   marital    45211 non-null  object
 3   education  43354 non-null  object
 4   default    45211 non-null  object
 5   balance    45211 non-null  int64 
 6   housing    45211 non-null  object
 7   loan       45211 non-null  object
 8   contact    32191 non-null  object
 9   day        45211 non-null  int64 
 10  month      45211 non-null  object
 11  duration   45211 non-null  int64 
 12  campaign   45211 non-null  int64 
 13  pdays      45211 non-null  int64 
 14  previous   45211 non-null  int64 
 15  y          45211 non-null  object
dtypes: int64(7), object(9)
memory usage: 5.5+ MB


In [10]:
#Replacing missing values in "job" and "education" with mode

for cols in ['job','education']:
    if df[cols].isnull().sum()>0:
        df[cols].fillna(df[cols].mode()[0],inplace=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 16 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   age        45211 non-null  int64 
 1   job        45211 non-null  object
 2   marital    45211 non-null  object
 3   education  45211 non-null  object
 4   default    45211 non-null  object
 5   balance    45211 non-null  int64 
 6   housing    45211 non-null  object
 7   loan       45211 non-null  object
 8   contact    32191 non-null  object
 9   day        45211 non-null  int64 
 10  month      45211 non-null  object
 11  duration   45211 non-null  int64 
 12  campaign   45211 non-null  int64 
 13  pdays      45211 non-null  int64 
 14  previous   45211 non-null  int64 
 15  y          45211 non-null  object
dtypes: int64(7), object(9)
memory usage: 5.5+ MB


In [11]:
#Replacing NaN in contact column with Unknown

df["contact"] = df["contact"].fillna("Unknown")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 16 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   age        45211 non-null  int64 
 1   job        45211 non-null  object
 2   marital    45211 non-null  object
 3   education  45211 non-null  object
 4   default    45211 non-null  object
 5   balance    45211 non-null  int64 
 6   housing    45211 non-null  object
 7   loan       45211 non-null  object
 8   contact    45211 non-null  object
 9   day        45211 non-null  int64 
 10  month      45211 non-null  object
 11  duration   45211 non-null  int64 
 12  campaign   45211 non-null  int64 
 13  pdays      45211 non-null  int64 
 14  previous   45211 non-null  int64 
 15  y          45211 non-null  object
dtypes: int64(7), object(9)
memory usage: 5.5+ MB


### 3.) Outlier Detection & Handling

In [12]:
#Function To Detect Outliers
def outlier_handle(df,columns):
    q1 = df[columns].quantile(0.25)
    q3 = df[columns].quantile(0.75)
    IQR = q3-q1

    lower_bound = q1 - 1.5*IQR
    upper_bound = q3 + 1.5*IQR

    #Method to replace outliers with min and max values
    df[columns] = np.where(df[columns] < lower_bound,lower_bound,df[columns])
    df[columns] = np.where(df[columns] > upper_bound,upper_bound,df[columns])
    return df

#Applying to numerical columns which is sensitive with outliers

for col in ['balance','campaign']:
    df = outlier_handle(df,col)


### Reason for applying only on **'balance'** amd **'campaign'** column is because:

- balance = *Extreme values distort the "average" customer profile.* (affects)
- campaign = *High values(e.g.,60calls)are effectively **noise** or errors.* (affects)
- previous = *"Outliers" here are the actual signal (returning customers).*
- duration = *Directly correlates with target; capping reduces predictive power.*
- age = *Distribution is normal; no extreme outliers to fix.*
    
    

### 4.) Transformations

In [16]:
#Create a flag for previous contact (-1 in 'pdays' means never contacted)

df['previously_contacted'] = np.where(df['pdays'] == -1, 0, 1)

### Savling Cleaned Data as csv

In [17]:
df.to_csv("C:/Users/Dell/OneDrive/Desktop/EDA PROJECT/Raw/cleaned_day2.csv", index=False)

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,y,previously_contacted
0,58,Management,Married,Tertiary,No,2143.0,Yes,No,Unknown,5,May,261,1.0,-1,0,No,0
1,44,Technician,Single,Secondary,No,29.0,Yes,No,Unknown,5,May,151,1.0,-1,0,No,0
2,33,Entrepreneur,Married,Secondary,No,2.0,Yes,Yes,Unknown,5,May,76,1.0,-1,0,No,0
3,47,Blue-collar,Married,Secondary,No,1506.0,Yes,No,Unknown,5,May,92,1.0,-1,0,No,0
4,33,Blue-collar,Single,Secondary,No,1.0,No,No,Unknown,5,May,198,1.0,-1,0,No,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45206,51,Technician,Married,Tertiary,No,825.0,No,No,Cellular,17,Nov,977,3.0,-1,0,Yes,0
45207,71,Retired,Divorced,Primary,No,1729.0,No,No,Cellular,17,Nov,456,2.0,-1,0,Yes,0
45208,72,Retired,Married,Secondary,No,3462.0,No,No,Cellular,17,Nov,1127,5.0,184,3,Yes,1
45209,57,Blue-collar,Married,Secondary,No,668.0,No,No,Telephone,17,Nov,508,4.0,-1,0,No,0
